In [ ]:
# imports
import numpy as np
import pandas as pd
import gzip                                                                                                                                                                              
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib import rcParams
rcParams["font.family"] = "Liberation Serif"

In [ ]:
# load the rubin DP2 objects from ECDFS:
# https://sdm-schemas.lsst.io/dp2.html#Object

df = pd.read_parquet("../../data/DP2_Object_ECDFS_highz.parquet")  

df.head(10)

In [ ]:
# plot histograms of mags

fig, axes = plt.subplots(1, 3, figsize=(15, 4))                                                                                                                                          
                                                                                                                                                                                        
for ax, col, color in zip(axes, ['u_cModelMag', 'g_cModelMag', 'r_cModelMag'], ['blue', 'green', 'red']):                                                                                
    ax.hist(df[col].dropna(), bins=50, color=color, alpha=0.7, edgecolor='black')
    ax.set_yscale('log')
    ax.set_xlabel(col)                                                                                                                                                                   
    ax.set_ylabel('Count')                                                                                                                                                             
    ax.set_title(col)
                                                                                                                                                                                        
plt.tight_layout()
plt.show() 

In [ ]:
# mag limits for DP2 https://dp2.lsst.io/overview/observations.html                                                                                                                                 
lim_u = 24.35                                                                                                                                                               
lim_g = 25.84                                                                                                                                                                         
lim_r = 25.47                                                                                                                                                                        
                                                                                                                                                                                        
mask = (df['u_cModelMag'] < lim_u) & (df['g_cModelMag'] < lim_g) & (df['r_cModelMag'] < lim_r)                                                                                           
df_filt = df[mask]
print(f"Removed {(~mask).sum()} / {len(df)} objects below mag limits, {mask.sum()} remaining")                                                                                           
                                                                                                                                                                                        
ug = df_filt['u_cModelMag'] - df_filt['g_cModelMag']
gr = df_filt['g_cModelMag'] - df_filt['r_cModelMag']                                                                                                                                     
                                                                                                                                                                                        
valid = np.isfinite(ug) & np.isfinite(gr)
n_missing = (~valid).sum()                                                                                                                                                               
print(f"{n_missing} / {len(df_filt)} objects missing color and excluded from plot")                                                                                                    

plt.figure(figsize=(7, 6))                                                                                                                                                               
plt.hist2d(gr[valid], ug[valid], bins=100, cmap='viridis', norm=colors.LogNorm())
cb = plt.colorbar()
cb.ax.tick_params(labelsize=18)
plt.xlabel('g - r', fontsize=20)                                                                                                                                                                    
plt.ylabel('u - g', fontsize=20)
plt.xticks(fontsize=18)                                                                                                                                                                  
plt.yticks(fontsize=18)
plt.xlim(-2,4)
plt.ylim(-3,4)
plt.tight_layout()
plt.show()

### Spec z galaxies

In [ ]:
# fetch spectroscopically classified galaxies from repository https://github.com/cosmosastro/speczcompilation

from astropy.table import Table

specz = Table.read("../../data/specz_compilation_COSMOS_DR1.1_unique.fits")

In [ ]:
# VUDS spectra



In [ ]:
# VVDS https://cesam.lam.fr/vvdspub/vvds_download.php  

# ZFLAG documented here: https://arxiv.org/pdf/1307.0545
# 4-> 100% prob correct; 3-> 95-100%; 2 -> 75-85%; 1 -> 50-75%; 0 -> no redshift could be assigned; 9 -> spectrum with single emission line, ~80% likely that z is correct

with gzip.open('../../data/cesam_vvds_spCDFS_DEEP.txt.gz', 'rt') as f:                                                                                                                   
    for i, line in enumerate(f):                                                                                                                                                       
        if i == 8:                                                                                                                                                                       
            cols = line.strip().lstrip('# ').split()                                                                                                                                   
            break

df_vvds = pd.read_csv('../../data/cesam_vvds_spCDFS_DEEP.txt.gz', compression='gzip', sep=r'\s+', skiprows=9, names=cols)            
total = len(df_vvds)
# we will cut out ZFLAG entries with value 1 or 0
df_vvds = df_vvds[df_vvds['ZFLAGS'].isin([4, 3, 2, 9])]                                                                                                                                  
print(f"{len(df_vvds)} / {total} objects remaining after ZFLAGS cut") 

df_vvds.head()

### rubin objects with specz 

In [ ]:
# create master dataframe to add matches to
df_rubin = df_filt[valid].reset_index(drop=True)
df_rubin_plus_specz = df_rubin.copy()

# add columns now to be fillted in with specz matches
df_rubin_plus_specz['z'] = np.nan
df_rubin_plus_specz['z_unc'] = np.nan                                                                                                                                                    
df_rubin_plus_specz['z_source'] = ''
df_rubin_plus_specz['z_source_id'] = ''

In [ ]:
#function to calculate distance between two ra/dec positions

DEGRA = np.pi / 180.0
def great_circle_distance(ra1_deg, dec1_deg, ra2_deg, dec2_deg):
    """
        Distance between two points on the sphere
    :param ra1_deg:
    :param dec1_deg:
    :param ra2_deg:
    :param dec2_deg:
    :return: distance in degrees
    """
    # this is orders of magnitude faster than astropy.coordinates.Skycoord.separation

    ra1, dec1, ra2, dec2 = (
        ra1_deg * DEGRA,
        dec1_deg * DEGRA,
        ra2_deg * DEGRA,
        dec2_deg * DEGRA,
    )
    delta_ra = np.abs(ra2 - ra1)
    distance = np.arctan2(
        np.sqrt(
            (np.cos(dec2) * np.sin(delta_ra)) ** 2
            + (
                np.cos(dec1) * np.sin(dec2)
                - np.sin(dec1) * np.cos(dec2) * np.cos(delta_ra)
            )
            ** 2
        ),
        np.sin(dec1) * np.sin(dec2) + np.cos(dec1) * np.cos(dec2) * np.cos(delta_ra),
    )

    return distance * 180.0 / np.pi

In [ ]:
# match rubin objects with VVDS catalog

from scipy.spatial import cKDTree

match_radius_arcsec = 1.0                                                                                                                                                                
match_radius_deg = match_radius_arcsec / 3600.0
                                                                                                                                                                                    
df_vvds_reset = df_vvds.reset_index(drop=True)                                                                                                                                           

# Build KDTree on Rubin coords for fast candidate lookup                                                                                                                                 
rubin_coords = np.deg2rad(np.c_[df_rubin['coord_ra'].values, df_rubin['coord_dec'].values])                                                                                            
vvds_coords  = np.deg2rad(np.c_[df_vvds_reset['ALPHA'].values,  df_vvds_reset['DELTA'].values])                                                                                          
                                                                                                                                                                                    
tree = cKDTree(rubin_coords)
# Query with chord distance approximation; refine with great_circle_distance                                                                                                             
chord_radius = 2 * np.sin(np.deg2rad(match_radius_deg) / 2)                                                                                                                              
idxs = tree.query_ball_point(vvds_coords, r=chord_radius)                                                                                                                                
                                                                                                                                                                                    
matches = []                                                                                                                                                                             
for i, candidates in enumerate(idxs):                                                                                                                                                  
    if not candidates:                                                                                                                                                                   
        continue
    dists = great_circle_distance(                                                                                                                                                       
        df_vvds_reset.loc[i, 'ALPHA'], df_vvds_reset.loc[i, 'DELTA'],                                                                                                                  
        df_rubin.loc[candidates, 'coord_ra'].values,                                                                                                                                     
        df_rubin.loc[candidates, 'coord_dec'].values,
    )                                                                                                                                                                                    
    best = np.argmin(dists)                                                                                                                                                            
    if dists[best] <= match_radius_deg:
        matches.append({                                                                                                                                                                 
            'vvds_idx': i,
            'rubin_idx': candidates[best],                                                                                                                                               
            'sep_arcsec': dists[best] * 3600.0,                                                                                                                                        
        })

df_matches = pd.DataFrame(matches)                                                                                                                                                       
print(f"{len(df_matches)} crossmatches found within {match_radius_arcsec}\"") 

print(f"{df_matches['rubin_idx'].duplicated().sum()} Rubin objects matched to multiple VVDS sources")                                                                                    
print(f"{df_matches['vvds_idx'].duplicated().sum()} VVDS objects matched to multiple Rubin sources")

rubin_idxs = df_matches['rubin_idx'].values                                                                                                                                              
vvds_idxs  = df_matches['vvds_idx'].values
                                                                                                                                                                                        
df_rubin_plus_specz.loc[rubin_idxs, 'z']           = df_vvds_reset.loc[vvds_idxs, 'Z'].values                                                                                            
df_rubin_plus_specz.loc[rubin_idxs, 'z_unc']       = df_vvds_reset.loc[vvds_idxs, 'ZFLAGS'].values
df_rubin_plus_specz.loc[rubin_idxs, 'z_source']    = 'vvds'                                                                                                                              
df_rubin_plus_specz.loc[rubin_idxs, 'z_source_id'] = df_vvds_reset.loc[vvds_idxs, 'ID-IAU'].values                                                                                       

print(f"{df_rubin_plus_specz['z'].notna().sum()} / {len(df_rubin_plus_specz)} objects with specz")                                                                                       
df_rubin_plus_specz[df_rubin_plus_specz['z'].notna()].head()

In [ ]:
# match rubin objects with VUDS catalog



In [ ]:
# visualize matches on color-color plot

ug_all = df_filt['u_cModelMag'] - df_filt['g_cModelMag']                                                                                                                                 
gr_all = df_filt['g_cModelMag'] - df_filt['r_cModelMag']                                                                                                                               
valid_all = np.isfinite(ug_all) & np.isfinite(gr_all)                                                                                                                                    
                                                                                                                                                                                        
has_z = df_rubin_plus_specz['z'].notna()                                                                                                                                                 
ug_z = df_rubin_plus_specz.loc[has_z, 'u_cModelMag'] - df_rubin_plus_specz.loc[has_z, 'g_cModelMag']                                                                                   
gr_z = df_rubin_plus_specz.loc[has_z, 'g_cModelMag'] - df_rubin_plus_specz.loc[has_z, 'r_cModelMag']                                                                                     
zvals = df_rubin_plus_specz.loc[has_z, 'z']                                                                                                                                            
                                                                                                                                                                                        
fig, ax = plt.subplots(figsize=(15, 13))                                                                                                                                                 

#background density map of galaxy counts from Rubin sample                                                                                                                                                                               
ax.hist2d(gr_all[valid_all], ug_all[valid_all], bins=100,                                                                                                                              
        cmap='Greys', norm=colors.LogNorm(), alpha=0.8)

#on top plot density map of spec-z matched galaxies colored by redshift                                                                                                                                                                                        
# sc = ax.scatter(gr_z, ug_z, c=zvals, cmap='plasma', s=10, zorder=5, vmin=0, vmax=10)
# cb = plt.colorbar(sc, ax=ax)                                                                                                                                                             
# cb.set_label('redshift', fontsize=18)
# cb.ax.tick_params(labelsize=18)
sc = ax.scatter(gr_z, ug_z, c=zvals, cmap='RdBu_r', s=60, zorder=5,
                norm=colors.TwoSlopeNorm(vmin=0, vcenter=1, vmax=3))
cb = plt.colorbar(sc, ax=ax)                                                                                                                                                             
cb.set_label('redshift', fontsize=18)
cb.ax.tick_params(labelsize=18)

                                                                                                                                                                                                                                                                                                                                
ax.set_xlabel('g - r', fontsize=20)
ax.set_ylabel('u - g', fontsize=20)                                                                                                                                                      
ax.tick_params(labelsize=18)                                                                                                                                                           
ax.set_xlim(-1, 3)
ax.set_ylim(-1, 3)
plt.tight_layout()
plt.show()  

### inspect spectra

In [ ]:
# plot a spectrum

LYMAN_ALPHA_REST = 1216.0  # Angstroms

RUBIN_BANDS = {
    "u": (3200, 4000,  "violet"),
    "g": (4000, 5520,  "royalblue"),
    "r": (5520, 6910,  "green"),
    "i": (6750, 8330,  "goldenrod"),
    "z": (8030, 9280,  "orange"),
    "y": (9280, 10800, "tomato"),
}

def plot_spectrum(rec, show_rubin_bands=True):  
    wave = np.array(rec.wavelength)
    flux = np.array(rec.flux)
    ivar = np.array(rec.ivar)
    z = rec.redshift

    err = np.where(ivar > 0, 1.0 / np.sqrt(ivar), np.nan)

    fig, ax = plt.subplots(figsize=(12, 3))

    if show_rubin_bands:
        for band, (lam_lo, lam_hi, color) in RUBIN_BANDS.items():
            ax.axvspan(lam_lo, lam_hi, alpha=0.08, color=color)

    ax.plot(wave, flux, lw=0.6, color="steelblue")
    ax.fill_between(wave, flux - err, flux + err, alpha=0.3, color="steelblue")
    ax.axhline(0, color="k", lw=0.5, ls="--")

    ax.axvline(LYMAN_ALPHA_REST, color="gray", lw=1.0, ls="--", label=f"Lyα rest ({LYMAN_ALPHA_REST:.0f} Å)")
    ax.axvline(LYMAN_ALPHA_REST * (1 + z), color="tomato", lw=1.0, ls="--", label=f"Lyα redshifted ({LYMAN_ALPHA_REST * (1 + z):.0f} Å)")

    ax.set_xlabel("Wavelength (Å)")
    ax.set_ylabel(r"Flux ($10^{-17}$ erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)")
    ax.set_title(f"z={z:.4f}  ra={rec.ra:.5f}  dec={rec.dec:.5f}")
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()